# Draft Jawaban — Tugas 1: Eksplorasi & Perancangan Arsitektur BI

**Studi kasus:** Kepatuhan Pajak Kendaraan Bermotor — Bapenda Provinsi Jawa Barat

> Notebook ini **draft jawaban isi**, bukan format pengumpulan resmi (template
> minta `.docx`, dikumpul via Gform — lihat catatan penutup). Bagian **Identitas**
> (Nama/NIM/Kelas/Kelompok) sengaja **tidak diisi** di sini — silakan lengkapi
> lalu pindahkan isinya ke `[TEMPLATE TUGAS 1] - Eksplorasi & Pemetaan Arsitektur BI.docx`.
>
> Semua isi di bawah ini ditarik dari pekerjaan ETL yang sudah benar-benar
> dibangun di repo ini (`etl/`, `config/sources.yaml`, `docs/data_dictionary.md`,
> `docs/katalog_analisis.md`) — bukan jawaban template generik. Edit bebas sesuai
> gaya bahasa kelompok.

## 1. Eksplorasi Implementasi BI

### 1.1 Profil Singkat Organisasi/Instansi

Badan Pendapatan Daerah (Bapenda) Provinsi Jawa Barat adalah organisasi
perangkat daerah yang bertugas melaksanakan urusan pemerintahan di bidang
pendapatan daerah, termasuk pemungutan Pajak Kendaraan Bermotor (PKB) dan Bea
Balik Nama Kendaraan Bermotor (BBNKB). Bapenda mengoperasikan puluhan kantor
Cabang Pelayanan (Samsat) yang tersebar di seluruh kabupaten/kota di Jawa
Barat, melayani jutaan kendaraan terdaftar setiap tahunnya. Kami memilih
Bapenda sebagai studi kasus karena datanya terbuka secara publik lewat portal
Satu Data Jabar, skalanya besar dan relevan secara fiskal (jutaan unit
kendaraan, triliunan rupiah nilai pajak per tahun), dan isu kepatuhan daftar
ulang pajak berdampak langsung pada pendapatan asli daerah.

### 1.3 Permasalahan

Seberapa besar potensi kehilangan pendapatan daerah akibat kendaraan yang
tidak/belum melakukan daftar ulang pajak setiap tahun, dan cabang pelayanan
maupun jenis kendaraan mana yang paling berkontribusi terhadap ketidakpatuhan
tersebut? Saat ini data jumlah kendaraan, data ketidakpatuhan, dan data
penerimaan pajak tersebar di dataset terpisah dengan skema kolom yang tidak
selalu konsisten, sehingga sulit dianalisis bersama secara langsung.
Implementasi BI dapat menjawab masalah ini dengan mengonsolidasikan
ketiga-empat dataset tersebut ke dalam satu star schema, sehingga tren
kepatuhan per tahun/wilayah/jenis kendaraan dapat dipantau dan potensi
kehilangan pendapatan dapat diestimasi secara konsisten dan berulang
(reproducible).

### 1.4 Sumber Referensi

- Observasi langsung terhadap dataset resmi Bapenda Provinsi Jawa Barat lewat
  API Satu Data Jabar ("Galura"): <https://galura.jabarprov.go.id>
  (organisasi `bapenda`, 17 dataset publik terkait kendaraan & pendapatan pajak)
- Portal katalog data resmi Provinsi Jawa Barat:
  <https://opendata.jabarprov.go.id/id/dataset>
- Halaman Open Data Bapenda: <https://bapenda.jabarprov.go.id/open-data-jabar/>
- Eksplorasi teknis internal kelompok: pipeline ETL yang sudah dibangun di
  repo ini (`etl/`, `config/sources.yaml`) sebagai bagian dari eksplorasi awal
  proyek, termasuk kamus data (`docs/data_dictionary.md`) dan katalog analisis
  (`docs/katalog_analisis.md`) yang sudah disusun dari data riil 2013–2025.

## 2. Perencanaan Arsitektur BI

### 2.1 Diagram Alur Arsitektur

Diagram di bawah memetakan 5 komponen standar arsitektur BI
(**Sumber Data → ETL → Data Warehouse → OLAP → Reporting/Dashboard**) ke
implementasi nyata proyek ini — bukan diagram konsep generik.

![Diagram Arsitektur BI](diagram_arsitektur_bi.png)

*(File gambar: `docs/diagram_arsitektur_bi.png` — sudah dimasukkan juga ke
`docs/Tugas1_Kelompok_Diagram-Arsitektur-BI.docx` di section 2.1.)*

In [ ]:
"""
OPSIONAL — jalankan cell ini kalau mau regenerate/modifikasi diagram
(ganti warna, label, atau tambah keterangan). Perlu Pillow:
    pip install pillow
Kalau cuma mau lihat diagramnya, cukup lihat gambar di cell markdown di atas
-- tidak perlu jalankan cell ini.
"""

from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

LEBAR, TINGGI = 2200, 950
SKALA = 2
W, H = LEBAR * SKALA, TINGGI * SKALA

BG = (255, 255, 255)
TEKS_GELAP = (30, 34, 45)
TEKS_ABU = (95, 102, 117)
GARIS = (150, 158, 171)

WARNA_BOX = [
    (219, 234, 254), (199, 224, 253), (191, 209, 250),
    (206, 199, 248), (224, 204, 250),
]
WARNA_TEPI = [
    (59, 130, 246), (37, 111, 234), (67, 97, 220),
    (109, 90, 210), (147, 87, 213),
]


def muat_font(nama_bold_italic, ukuran):
    """Coba beberapa lokasi font umum; fallback ke font default Pillow."""
    kandidat = [
        f"C:/Windows/Fonts/{nama_bold_italic}",
        f"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        f"/Library/Fonts/{nama_bold_italic}",
    ]
    for path in kandidat:
        if Path(path).exists():
            return ImageFont.truetype(path, ukuran * SKALA)
    return ImageFont.load_default()


f_judul = muat_font("calibrib.ttf", 30)
f_label = muat_font("calibrib.ttf", 24)
f_sub = muat_font("calibri.ttf", 16)
f_kecil = muat_font("calibrii.ttf", 15)
f_footer = muat_font("calibrii.ttf", 14)

img = Image.new("RGB", (W, H), BG)
d = ImageDraw.Draw(img)


def teks_center(cx, cy, teks, fnt, warna, max_lebar=None, spasi=8):
    baris_eksplisit = teks.split("\n")
    if max_lebar is None:
        baris = baris_eksplisit
    else:
        baris = []
        for line in baris_eksplisit:
            cur = ""
            for k in line.split(" "):
                coba = (cur + " " + k).strip()
                if d.textlength(coba, font=fnt) <= max_lebar:
                    cur = coba
                else:
                    if cur:
                        baris.append(cur)
                    cur = k
            if cur:
                baris.append(cur)
    tinggi_baris = fnt.size + spasi
    y = cy - (len(baris) * tinggi_baris) / 2
    for b in baris:
        lebar = d.textlength(b, font=fnt)
        d.text((cx - lebar / 2, y), b, font=fnt, fill=warna)
        y += tinggi_baris


judul = "Arsitektur BI \u2014 Kepatuhan Pajak Kendaraan Bermotor Jawa Barat"
d.text(((W - d.textlength(judul, font=f_judul)) / 2, 40 * SKALA), judul, font=f_judul, fill=TEKS_GELAP)
subjudul = "Studi kasus: Badan Pendapatan Daerah (Bapenda) Provinsi Jawa Barat"
d.text(((W - d.textlength(subjudul, font=f_sub)) / 2, 88 * SKALA), subjudul, font=f_sub, fill=TEKS_ABU)

# --- EDIT DI SINI kalau mau ubah isi tiap komponen ---
komponen = [
    ("Sumber Data", "API Satu Data Jabar\n(Bapenda, publik)", "4 dataset CSV agregat\ntahunan 2013\u20132025"),
    ("ETL", "Python + pandas", "extract \u2192 transform \u2192\nbuild_star \u2192 validate"),
    ("Data\nWarehouse", "Star schema", "1 fact + 4 dimensi\n(CSV / DuckDB)"),
    ("OLAP", "Agregasi SUM/GROUP BY", "tingkat kepatuhan,\nestimasi pajak hilang"),
    ("Reporting /\nDashboard", "Tableau / notebook", "(tahap berikutnya,\nbelum dikerjakan)"),
]

n = len(komponen)
margin_x = 70 * SKALA
gap_panah = 46 * SKALA
box_atas = 170 * SKALA
box_tinggi = 430 * SKALA
box_lebar = (W - 2 * margin_x - gap_panah * (n - 1)) / n

centers = []
for i, (label, impl, detail) in enumerate(komponen):
    x0 = margin_x + i * (box_lebar + gap_panah)
    x1 = x0 + box_lebar
    y0, y1 = box_atas, box_atas + box_tinggi
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    centers.append((x1, cy))

    d.rounded_rectangle((x0, y0, x1, y1), radius=22 * SKALA, fill=WARNA_BOX[i], outline=WARNA_TEPI[i], width=3 * SKALA)

    chip_r = 22 * SKALA
    chip_cy = y0 + 34 * SKALA
    d.ellipse((cx - chip_r, chip_cy - chip_r, cx + chip_r, chip_cy + chip_r), fill=WARNA_TEPI[i])
    num = str(i + 1)
    d.text((cx - d.textlength(num, font=f_label) / 2, chip_cy - f_label.size / 2 - 4 * SKALA), num, font=f_label, fill=(255, 255, 255))

    teks_center(cx, y0 + 95 * SKALA, label, f_label, TEKS_GELAP, max_lebar=box_lebar - 24 * SKALA, spasi=6)

    ly = y0 + 150 * SKALA
    d.line((x0 + 20 * SKALA, ly, x1 - 20 * SKALA, ly), fill=WARNA_TEPI[i], width=2 * SKALA)

    teks_center(cx, ly + 40 * SKALA, impl, f_sub, TEKS_GELAP, max_lebar=box_lebar - 24 * SKALA, spasi=6)
    teks_center(cx, y1 - 80 * SKALA, detail, f_kecil, TEKS_ABU, max_lebar=box_lebar - 20 * SKALA, spasi=6)

for i in range(n - 1):
    x_start, cy = centers[i]
    x_end = x_start + gap_panah
    d.line((x_start + 6 * SKALA, cy, x_end - 14 * SKALA, cy), fill=GARIS, width=3 * SKALA)
    ah = 9 * SKALA
    d.polygon([(x_end - 14 * SKALA, cy - ah), (x_end, cy), (x_end - 14 * SKALA, cy + ah)], fill=GARIS)

footer = "Diagram digambar berdasarkan implementasi nyata (repo ETL) \u2014 bukan diagram konsep generik"
d.text(((W - d.textlength(footer, font=f_footer)) / 2, H - 50 * SKALA), footer, font=f_footer, fill=TEKS_ABU)

img = img.resize((LEBAR, TINGGI), Image.LANCZOS)
img.save("diagram_arsitektur_bi.png", "PNG")
img

### 2.2 Tantangan yang Mungkin Dihadapi

Lima tantangan berikut **bukan tantangan generik dari textbook** — semuanya
ditemukan langsung saat membangun pipeline ETL di repo ini.

**Sumber Data**
Portal katalog resmi (`opendata.jabarprov.go.id`) berada di belakang
Cloudflare dan menolak request otomatis (403/CF Challenge), sehingga extract
tidak bisa dilakukan lewat scraping langsung. Solusinya ditemukan lewat API
publik alternatif resmi (Satu Data Jabar/"Galura") yang tidak
terdokumentasikan secara publik strukturnya — perlu eksplorasi endpoint
manual (`/api/organisasi`, `/api/datasets`, `/api/bigdata/<schema>/<table>`)
sebelum data bisa diambil terprogram.

**ETL**
Skema kolom tidak konsisten antar dataset yang seharusnya sejenis (mis.
`jenis_pkb` di satu dataset vs `jenis_pajak` di dataset lain untuk konsep yang
sama). Lebih rumit lagi, satu kategori kendaraan berganti nama di tengah
rentang data historis — `"Sedan, Jeep, Minibus"` berubah jadi
`"Sedan, Jeep, Station Wagon"` persis mulai tahun 2022 di salah satu dataset,
sementara dataset lain tetap memakai nama lama — butuh normalisasi manual
supaya grain tetap bisa disambungkan.

**Data Warehouse**
Grain geografis harus diputuskan dengan hati-hati: dataset kepatuhan memakai
satuan "cabang pelayanan" (Samsat), bukan kabupaten/kota, dan satu cabang bisa
melayani wilayah administratif yang tidak sama persis dengan satu
kabupaten/kota. Memaksakan join ke level kabupaten/kota akan menggandakan
baris fact, sehingga kabupaten/kota harus diperlakukan sebagai atribut
opsional pada dimensi wilayah, bukan grain fact.

**OLAP**
Definisi kategori "Alat Berat" tampak tidak konsisten antar dataset sumber:
total kendaraan alat berat yang dilaporkan sangat kecil (puluhan unit per
tahun se-provinsi), namun angka "tidak daftar ulang" untuk kategori yang sama
jauh lebih besar di beberapa cabang — mengindikasikan definisi kategori yang
berbeda antar dataset. Segmen ini perlu ditandai tidak dapat dipercaya untuk
perhitungan rasio kepatuhan sampai diverifikasi ke sumber datanya.

**Reporting/Dashboard**
Belum dikerjakan pada tahap ini. Tantangan yang diperkirakan: memilih tool
yang sesuai kebutuhan kelompok (Tableau/Power BI/notebook Python), dan
memastikan measure rasio (`tingkat_kepatuhan`) tidak di-*average* secara
keliru saat diagregasi di level dashboard — harus dihitung ulang sebagai
`SUM(jumlah_daftar_ulang) / SUM(jumlah_kendaraan_total)` di tiap level
agregasi, bukan dirata-ratakan langsung dari baris fact.

---

## Catatan Penutup

- Notebook ini **draft isi**, bukan format submission. Format resmi Tugas 1:
  file `.docx` (`Tugas1_Kelompok_Nama.docx`), font Times New Roman 12pt /
  Calibri 11pt spasi 1,15, dikumpulkan lewat Google Form sebelum Pertemuan 3.
- Diagram di section 2.1 **sudah** disisipkan ke
  `docs/Tugas1_Kelompok_Diagram-Arsitektur-BI.docx` — tinggal salin teks di
  atas ke bagian 1.1/1.3/1.4/2.2 template tersebut, isi Identitas, cek ulang
  gaya bahasa, lalu ganti nama file sesuai ketentuan sebelum submit.
- Detail teknis lengkap (skema kolom, aturan transform, keterbatasan data) ada
  di [`docs/data_dictionary.md`](data_dictionary.md) dan
  [`docs/katalog_analisis.md`](katalog_analisis.md) kalau perlu kutip lebih
  jauh untuk bagian mana pun di laporan.